# 1. Парсинг данных с сайта - https://rkn.gov.ru/activity/connection/register/license/

In [13]:
pip install selenium pandas

Note: you may need to restart the kernel to use updated packages.


In [117]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time
import re

In [17]:
# Настройка браузера
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
driver = webdriver.Chrome(options=options)

In [65]:
# Забираем ссылки на карточки (со всех листов)

base_url = "https://rkn.gov.ru/activity/connection/register/license/p"
page_number = 0  # начинаем с нуля

all_links = []

while True:
    url = f"{base_url}{page_number}/"
    print(f"Страница в работе: p{page_number}")

    driver.get(url)
    time.sleep(1)

    licenses = driver.find_elements(By.CSS_SELECTOR, 'tr.clmn1, tr.clmn2')

    if not licenses:
        print(f"Страница p{page_number} пуста. Останавливаемся.")
        break

    for lic in licenses:
        try:
            link = lic.find_element(By.TAG_NAME, 'a').get_attribute('href')
            all_links.append(link)
        except Exception as e:
            print(f"Ошибка при сборе ссылки: {e}")

    page_number += 100  # прибавляем 100 для следующей страницы

Страница в работе: p0
Страница в работе: p100
Страница в работе: p200
Страница в работе: p300
Страница в работе: p400
Страница в работе: p500
Страница в работе: p600
Страница в работе: p700
Страница в работе: p800
Страница в работе: p900
Страница в работе: p1000
Страница в работе: p1100
Страница в работе: p1200
Страница в работе: p1300
Страница в работе: p1400
Страница в работе: p1500
Страница в работе: p1600
Страница в работе: p1700
Страница в работе: p1800
Страница в работе: p1900
Страница в работе: p2000
Страница в работе: p2100
Страница в работе: p2200
Страница в работе: p2300
Страница в работе: p2400
Страница в работе: p2500
Страница в работе: p2600
Страница в работе: p2700
Страница в работе: p2800
Страница в работе: p2900
Страница в работе: p3000
Страница в работе: p3100
Страница в работе: p3200
Страница в работе: p3300
Страница в работе: p3400
Страница в работе: p3500
Страница в работе: p3600
Страница в работе: p3700
Страница в работе: p3800
Страница в работе: p3900
Страница в р

In [68]:
# Собираем информацию с карточек 

data = []

total_links = len(all_links)  # всего ссылок

print(f"Всего лицензий для обработки: {total_links}")

for idx, link in enumerate(all_links, 1):  # enumerate даёт номер итерации
    print(f"Обрабатываю {idx}/{total_links}: {link}")  # вывод прогресса

    driver.execute_script("window.open(arguments[0]);", link)
    driver.switch_to.window(driver.window_handles[1])

    try:
        time.sleep(1)  # ждём загрузку карточки

        # Находим ВСЕ строки в таблице
        rows = driver.find_elements(By.CSS_SELECTOR, 'table.TblList tr')
        record = {}

        for row in rows:
            tds = row.find_elements(By.TAG_NAME, 'td')
            if len(tds) >= 2:
                key = tds[0].text.strip()  # Первая ячейка — название поля
                value = tds[1].text.strip()  # Вторая ячейка — значение поля
                record[key] = value

        if record:
            data.append(record)

    except Exception as e:
        print(f"Ошибка при парсинге карточки: {e}")

    # Закрываем вкладку карточки
    driver.close()
    driver.switch_to.window(driver.window_handles[0])

Всего лицензий для обработки: 26229
Обрабатываю 1/26229: https://rkn.gov.ru/activity/connection/register/license/p0/?id=%D0%9B030-00114-77%2F00070257
Обрабатываю 2/26229: https://rkn.gov.ru/activity/connection/register/license/p0/?id=%D0%9B030-00114-77%2F00073326
Обрабатываю 3/26229: https://rkn.gov.ru/activity/connection/register/license/p0/?id=%D0%9B030-00114-77%2F00069417
Обрабатываю 4/26229: https://rkn.gov.ru/activity/connection/register/license/p0/?id=%D0%9B030-00114-77%2F00070479
Обрабатываю 5/26229: https://rkn.gov.ru/activity/connection/register/license/p0/?id=%D0%9B030-00114-77%2F00072310
Обрабатываю 6/26229: https://rkn.gov.ru/activity/connection/register/license/p0/?id=%D0%9B030-00114-77%2F00083561
Обрабатываю 7/26229: https://rkn.gov.ru/activity/connection/register/license/p0/?id=%D0%9B030-00114-77%2F00083562
Обрабатываю 8/26229: https://rkn.gov.ru/activity/connection/register/license/p0/?id=%D0%9B030-00114-77%2F00077959
Обрабатываю 9/26229: https://rkn.gov.ru/activity/con

In [71]:
# переводим в df
df = pd.DataFrame(data)

In [73]:
# сохраняем в csv
df.to_csv('Лицензии.csv', index = False)

# Работа с DF. Обработка собранной информации

In [75]:
print([columns for columns in df])

['Статус лицензии', 'Наименование лицензирующего органа', 'Регистрационный номер лицензии', 'Номер лицензии, присвоенный до 01.03.2022', 'Дата предоставления лицензии', 'День начала оказания услуг', 'Срок действия до', 'Полное наименование лицензиата', 'Сокращенное наименование', 'Фирменное наименование', 'Организационно-правовая форма', 'Адрес места нахождения', 'ОГРН', 'ИНН', 'Номер телефона', 'Адрес электронной почты', 'Территория действия лицензии', 'Лицензируемый вид деятельности с указанием выполняемых работ, составляющих лицензируемый вид деятельности', 'Номер и дата лицензионного приказа о предоставлении лицензии', 'Номер и дата приказа Роскомнадзора о переоформлении лицензии', 'Номер и дата приказа Роскомнадзора о продлении срока действия лицензии', 'Номер и дата приказа Роскомнадзора о приостановлении/возобновлении действия лицензии', 'Номер и дата приказа лицензирующего органа о прекращении действия лицензии, основание и дата прекращения действия лицензии', 'Основание, дата 

In [77]:
pd.set_option('display.max_columns', 500)

In [82]:
# Создаем df инфа про лицензии
df_License = df.copy()
df_License = df_License[[
    'Регистрационный номер лицензии', 'Статус лицензии', 'День начала оказания услуг', 'Срок действия до',
    'ИНН', 'Территория действия лицензии', 'Лицензируемый вид деятельности с указанием выполняемых работ, составляющих лицензируемый вид деятельности', 
]]

In [87]:
df_company = df.copy()
df_company = df_company[[
    'Полное наименование лицензиата', 'Сокращенное наименование', 'Организационно-правовая форма', 'Адрес места нахождения',
    'ОГРН', 'ИНН', 'Номер телефона', 'Адрес электронной почты'
]]

In [93]:
df_company = df_company.drop_duplicates()

In [96]:
df_License = df_License.drop_duplicates()

In [86]:
df['Фирменное наименование'].unique()

array(['', nan], dtype=object)

In [99]:
# ИНН, которых больше 1
df_company['ИНН'].value_counts().sort_values(ascending=False)

7801002274    3
1605003947    3
8602060555    3
1300012383    3
7703119524    3
             ..
7805071261    1
7706412930    1
6228049685    1
1308080160    1
9001007809    1
Name: ИНН, Length: 8938, dtype: int64

In [102]:
# Данные не разнятся. Скорее всего где-то заполнили немного подругому
df_company[df_company['ИНН'] == '1300012383']

,Полное наименование лицензиата,Сокращенное наименование,Организационно-правовая форма,Адрес места нахождения,ОГРН,ИНН,Номер телефона,Адрес электронной почты
2194,ГОСУДАРСТВЕННОЕ АВТОНОМНОЕ УЧРЕЖДЕНИЕ РЕСПУБЛИ...,"ГАУ РЕСПУБЛИКИ МОРДОВИЯ ""ЦЦР""",Государственные автономные учреждения субъекто...,"430016, РЕСПУБЛИКА МОРДОВИЯ, городской округ С...",1241300003108,1300012383,(8342)39-10-00\n[внести изменения],boa@e-mordovia.ru\n[внести изменения]
2196,ГОСУДАРСТВЕННОЕ АВТОНОМНОЕ УЧРЕЖДЕНИЕ РЕСПУБЛИ...,"ГАУ РЕСПУБЛИКИ МОРДОВИЯ ""ЦЦР""",Государственные автономные учреждения субъекто...,"430016, РЕСПУБЛИКА МОРДОВИЯ, городской округ С...",1241300003108,1300012383,(8342)39-10-00\n[внести изменения],it@e-mordovia.ru\n[внести изменения]
2197,ГОСУДАРСТВЕННОЕ АВТОНОМНОЕ УЧРЕЖДЕНИЕ РЕСПУБЛИ...,"ГАУ РЕСПУБЛИКИ МОРДОВИЯ ""ЦЦР""",Государственные автономные учреждения субъекто...,"430016, РЕСПУБЛИКА МОРДОВИЯ, городской округ С...",1241300003108,1300012383,(8342) 39-10-00\n[внести изменения],it@e-mordovia.ru\n[внести изменения]


In [103]:
# Удалим дубликаты по ИНН, чтобы не было повторяющихся
df_company = df_company.drop_duplicates(subset = ['ИНН'])

In [105]:
df_company['ИНН'].value_counts().sort_values(ascending=False)

0317002737    1
6375997049    1
2360980415    1
6027201604    1
2722999113    1
             ..
7717022723    1
5254001230    1
7423000572    1
7717127211    1
9001007809    1
Name: ИНН, Length: 8938, dtype: int64

In [108]:
# Проверяем на пустые значения
df_company.isnull().sum()

Полное наименование лицензиата      0
Сокращенное наименование          882
Организационно-правовая форма     865
Адрес места нахождения            865
ОГРН                              865
ИНН                                 0
Номер телефона                      0
Адрес электронной почты             0
dtype: int64

In [112]:
# Логика заполнения пустых. Если сокращенного наименования нет - берем полное 
# Если организационной правовой формы нет / адреса / ОГРН - "Не установлено"
df_company = df_company.copy()
df_company.loc[:, 'Сокращенное наименование'] = df_company['Сокращенное наименование'].fillna(df_company['Полное наименование лицензиата'])
df_company.loc[:, 'Организационно-правовая форма'] = df_company['Организационно-правовая форма'].fillna('Не установлено')
df_company.loc[:, 'Адрес места нахождения'] = df_company['Адрес места нахождения'].fillna('Не установлено')
df_company.loc[:, 'ОГРН'] = df_company['ОГРН'].fillna('Не установлено')

In [113]:
# повторная проверка на пустые
df_company.isnull().sum()

Полное наименование лицензиата    0
Сокращенное наименование          0
Организационно-правовая форма     0
Адрес места нахождения            0
ОГРН                              0
ИНН                               0
Номер телефона                    0
Адрес электронной почты           0
dtype: int64

In [118]:
df_company['Регион'] = df_company['Адрес места нахождения'].str.extract(r'^[^,]*,\s*([^,]+)')

In [120]:
df_company['Регион'] = df_company['Регион'].str.lower()

In [124]:
# проверяем нулевые значения. Их нужно поправить
df_company[df_company['Регион'].isnull()]['Адрес места нахождения'].unique()

array(['Не установлено'], dtype=object)

In [126]:
df_company.loc[:, 'Регион'] = df_company['Регион'].fillna('Не установлено')

In [130]:
df_company['Номер телефона'] = df_company['Номер телефона'].str.replace('\n[внести изменения]', '', regex=False)
df_company['Адрес электронной почты'] = df_company['Адрес электронной почты'].str.replace('\n[внести изменения]', '', regex=False)

In [133]:
# Проверяем на пустые значения
df_License.isnull().sum()

Регистрационный номер лицензии                                                                               0
Статус лицензии                                                                                              0
День начала оказания услуг                                                                                   0
Срок действия до                                                                                             0
ИНН                                                                                                          0
Территория действия лицензии                                                                                 0
Лицензируемый вид деятельности с указанием выполняемых работ, составляющих лицензируемый вид деятельности    0
dtype: int64

In [135]:
df_License['Территория действия лицензии'].unique()

array(['Республика Бурятия: Северобайкальск г',
       'Республика Дагестан: Дубки, Чиркей', 'Иркутская область', ...,
       'Ленинградская область: Выборгский р-н, Выборг г',
       'Свердловская область: Кушва г (п.у.п. - Кушва г, гора Малая Благодатка), Новолялинский р-н, Новая Ляля г',
       'Запорожская область: Васильевский район, город Энергодар'],
      dtype=object)

In [142]:
df_License = df_License.copy()
df_License['Регион_Лицензии'] = df_License['Территория действия лицензии'].str.extract(r'^([^:]+)')

In [144]:
df_License['Регион_Лицензии'].unique()

array(['Республика Бурятия', 'Республика Дагестан', 'Иркутская область',
       'Российская Федерация', 'Ставропольский край',
       'Калужская область; Московская область; Тульская область',
       'Московская область', 'Мурманская область',
       'Ямало-Ненецкий автономный округ', 'Москва',
       'Нижегородская область', 'Республика Татарстан (Татарстан)',
       'Самарская область', 'Тверская область', 'Хабаровский край',
       'Псковская область', 'Краснодарский край', 'Тамбовская область',
       'Кемеровская область - Кузбасс', 'Кемеровская область',
       'Челябинская область', 'Республика Хакасия',
       'Ханты-Мансийский автономный округ - Югра', 'Курская область',
       'Республика Мордовия', 'Тюменская область',
       'Чувашская Республика - Чувашия', 'Республика Саха (Якутия)',
       'Москва; Московская область', 'Сахалинская область',
       'Приморский край', 'Республика Адыгея (Адыгея)',
       'Чукотский автономный округ', 'Кировская область',
       'Санкт-Пет

In [147]:
# Сделаем отдельный df для региона действия
df_License_region = df_License.copy()
df_License_region = df_License_region[[
    'Регистрационный номер лицензии', 'Регион_Лицензии'
]]

In [161]:
# разделяем чтобы один регион одна строка
df_License_region = df_License_region.copy()

df_License_region_1 = (
    df_License_region.assign(Регион_Лицензии=df_License_region['Регион_Лицензии'].str.split(';'))
    .explode('Регион_Лицензии')
)

# Убираем лишние пробелы, если есть
df_License_region_1['Регион_Лицензии'] = df_License_region_1['Регион_Лицензии'].str.strip()

In [166]:
# проверяем пустые значения
df_License[df_License['Регион_Лицензии'].isnull()]

,Регистрационный номер лицензии,Статус лицензии,День начала оказания услуг,Срок действия до,ИНН,Территория действия лицензии,"Лицензируемый вид деятельности с указанием выполняемых работ, составляющих лицензируемый вид деятельности",Регион_Лицензии
7091,Л030-00114-77/00060526,действующая,19.02.2024,20.10.2028,0914000290,,Услуги связи для целей эфирного вещания,NaN
16577,Л030-00114-77/00071859,действующая,16.12.2024,21.07.2029,5911997011,,Услуги связи для целей эфирного вещания,NaN


In [167]:
df_License_region_1.loc[:, 'Регион_Лицензии'] = df_License_region_1['Регион_Лицензии'].fillna('Не установлено')

In [ ]:
# Финальные df проекта:

# df_License_region_1 — информация о регионах действия лицензий.
# Внешний ключ — Регистрационный номер лицензии (для объединения с df_License).
# Каждая строка — отдельный регион по одной лицензии.

# df_License — информация по лицензиям (статус, даты, ИНН, вид деятельности).
# Первичный ключ — Регистрационный номер лицензии.
# Внешний ключ — ИНН (ссылается на df_company, т.е. компания, которой выдана лицензия).

# df_company — информация о компаниях (название, форма, ОГРН, ИНН и т.д.).
# Первичный ключ — ИНН.

In [173]:
df_License_region_1.to_csv('df_License_region_1.csv', index = False, sep = ';', encoding = 'cp1251')
df_License.to_csv('df_License.csv', index = False, sep = ';', encoding = 'cp1251')
df_company.to_csv('df_company.csv', index = False, sep = ';', encoding = 'cp1251')